## Setup

In [1]:
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

True

## MCP tools and agents

In [ ]:
# Mounts the CultPass and UDA-Hub MCP servers in memory (no processes, no network).
# The tools are async-only, so agents must be run with `await agent.ainvoke(...)`.
from agentic.mcp_servers.toolbox import build_default_toolbox
from agentic.agents.registry import build_agents

toolbox = build_default_toolbox()
await toolbox.connect()          # re-running this cell is safe; run the last cell to close it

In [ ]:
# Each agent only gets the tools bound to it in agentic/agents/registry.py
agents = build_agents(toolbox)
for key, agent in agents.items():
    print(f"{key:20s} {agent.get_agent_tools()}")

## Run

In [ ]:
# Design notes: agentic/design/architecture.md (TODO: diagram and agent roles, Rubric 2)
# Agents:       agentic/agents/ (tools bound in agentic/agents/registry.py)
# MCP tools:    agentic/mcp_servers/
# Orchestrator: agentic/workflow.py

In [ ]:
# The orchestrator needs the connected toolbox from the cells above, so it is built here.
from agentic.workflow import build_orchestrator

orchestrator = await build_orchestrator(toolbox)

In [4]:
await chat_interface(orchestrator, "1")

User: Hi
Assistant: Hello! How can I assist you today?
User: q
Assistant: Goodbye!


In [5]:
# Everything saved for conversation "1": one snapshot per step, newest first
config = {"configurable": {"thread_id": "1"}}
history = [snapshot async for snapshot in orchestrator.aget_state_history(config)]
history[0].values["messages"]

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='43c56100-7a4e-4ff0-adb7-1fbbfdac82e3'),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 60, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C1loRWd5jRqktu5Fut6YZAZsTdc6S', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--0ab48c4a-6130-4dc9-96cd-271e2be7b7c8-0', usage_metadata={'input_tokens': 60, 'output_tokens': 10, 'total_tokens': 70, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

## Test scenarios

In [ ]:
# TODO: Run >= 4 different ticket types end to end (Rubric 5)
# pseudocode:
#   scenarios = [
#     {"name": "simple FAQ",       "ticket": ..., "expect": "resolved from KB, no escalation"},
#     {"name": "technical issue",  "ticket": ..., "expect": "KB answer + maybe tool call"},
#     {"name": "urgent problem",   "ticket": ..., "expect": "high urgency routing / escalation"},
#     {"name": "no KB match",      "ticket": ..., "expect": "escalated (low confidence)"},
#   ]
#   for each scenario:
#       thread_id = unique ticket_id
#       result = orchestrator.invoke({"messages": [HumanMessage(ticket)]},
#                                    config={"configurable": {"thread_id": thread_id}})
#       print final answer, classification, route taken, confidence, escalated?
#       compare against `expect` (simple assert or table)

## Memory demo

In [ ]:
# TODO: Show short-term and long-term memory (Rubric 4)
# pseudocode:
#   short-term: invoke twice with the SAME thread_id; 2nd message refers to the 1st
#               ("what did I just ask?") and the answer must use the earlier turn
#   long-term : after ticket A is resolved, a memory-writer step saves resolution + preference
#               start a NEW thread_id for the same customer
#               show the agent's answer references the stored history / preference
#   inspect   : orchestrator.get_state_history(config) -> print messages and tool calls
#               for a given thread_id (see the cell above)

## Logs and metrics

In [ ]:
# TODO: Read the structured logs and summarise them (Rubric 5)
# pseudocode:
#   parse the JSON-lines log file written by the workflow
#   print counts:  KB retrieval succeeded vs escalated
#                  tool usage per tool name
#                  routing decisions per category
#   check that no email / full message body / secrets appear in the logs

## Cleanup

In [ ]:
await toolbox.aclose()